# ⟁ Artha — Fine-tuning on Google Colab
> Self-contained notebook. No repo needed. Just run all cells top to bottom.

In [1]:
# Cell 1 — Install dependencies
!pip install -q transformers peft datasets accelerate tokenizers
print('Done!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 12.5 MB/s eta 0:00:00
Done!


In [2]:
# Cell 2 — Setup
import os
os.makedirs('/content/data', exist_ok=True)
print('Directories ready')

Directories ready


In [3]:
# Cell 3 — Generate corpus (50k English-Artha pairs)
import json, random

TONES = [("formal","tone:formal"),("professional","tone:professional"),
         ("casual","tone:casual"),("friendly","tone:friendly"),
         ("academic","tone:academic"),("persuasive","tone:persuasive")]
FORMATS = [("bullet points","fmt:bullets"),("numbered list","fmt:list"),
           ("a table","fmt:table"),("JSON","fmt:json"),("markdown","fmt:md"),
           ("numbered steps","fmt:steps"),("a one liner","fmt:tldr"),("an outline","fmt:outline")]
LEVELS = [("a 5 year old","lvl:eli5"),("a beginner","lvl:beginner"),
          ("an intermediate","lvl:mid"),("an expert","lvl:expert")]
LANGS = [("French","lang:fr"),("Spanish","lang:es"),("German","lang:de"),
         ("Hindi","lang:hi"),("Chinese","lang:zh"),("Japanese","lang:ja")]
CODE_LANGS = [("Python","code:py"),("JavaScript","code:js"),("TypeScript","code:ts"),
              ("Rust","code:rs"),("Go","code:go"),("Java","code:java"),("SQL","code:sql")]
AUDIENCES = [("developers","@dev"),("managers","@mgr"),("clients","@client"),
             ("students","@student"),("executives","@exec")]
FILLERS = ["Please ","Could you ","Can you ","I want you to ",
           "Help me ","Kindly ","I'd like you to ","Would you "]
TOPICS = ["machine learning","climate change","quantum computing","blockchain",
          "neural networks","the stock market","renewable energy","cybersecurity",
          "product roadmap","sales data","customer feedback","API design"]
CONSTRAINTS_POS = [("focus on key facts","+facts"),("focus on performance","+perf"),
                   ("include examples","+examples"),("emphasise security","+security"),
                   ("prioritise clarity","+clarity")]
CONSTRAINTS_NEG = [("ignore opinions","-opinions"),("avoid jargon","-jargon"),
                   ("no fluff","-fluff"),("without passive voice","-passive"),
                   ("exclude style issues","-style")]

def r(lst): return random.choice(lst)
def maybe(lst, p=0.5): return r(lst) if random.random() < p else None
def filler(): return r(FILLERS) if random.random() < 0.7 else ""

def gen_summarise():
    doc = r(["doc","article","report","essay","blog post","readme"])
    n = r([2,3,4,5])
    fmt_en,fmt_ar = r(FORMATS[:5])
    pos = maybe(CONSTRAINTS_POS)
    neg = maybe(CONSTRAINTS_NEG)
    en = f"{filler()}summarise this {doc} in {n} {fmt_en}"
    ar = f"sum[{doc}](#{n}, {fmt_ar})"
    if pos: en += f", {pos[0]}"; ar += f" {pos[1]}"
    if neg: en += f", {neg[0]}"; ar += f" {neg[1]}"
    return {"english": en.strip(), "artha": ar.strip()}

def gen_generate():
    topic = r(TOPICS)
    fmt_en,fmt_ar = r(FORMATS)
    tone_en,tone_ar = r(TONES)
    aud_en,aud_ar = r(AUDIENCES)
    words = r([100,150,200,300])
    return {"english": f"{filler()}write a {tone_en} piece about {topic} for {aud_en}, format as {fmt_en}, under {words} words",
            "artha": f"gen[{topic}]({tone_ar}, {fmt_ar}, {aud_ar}, ~{words}w)"}

def gen_fix():
    lang_en,lang_ar = r(CODE_LANGS)
    explain = random.random() < 0.6
    en = f"{filler()}fix the bug in this {lang_en} code"
    ar = f"fix[code]({lang_ar})"
    if explain: en += " and explain what was wrong"; ar += " -> {diff+explain}"
    else: ar += " -> {diff}"
    return {"english": en, "artha": ar}

def gen_explain():
    topic = r(TOPICS)
    lvl_en,lvl_ar = r(LEVELS)
    fmt_en,fmt_ar = r(FORMATS[:6])
    return {"english": f"{filler()}explain {topic} to {lvl_en} using {fmt_en}",
            "artha": f"xpl[{topic}]({lvl_ar}, {fmt_ar})"}

def gen_compare():
    items = random.sample(TOPICS, 2)
    fmt_en,fmt_ar = r(FORMATS[:5])
    pos = maybe(CONSTRAINTS_POS)
    en = f"{filler()}compare {items[0]} and {items[1]}, format as {fmt_en}"
    ar = f"cmp[{items[0]}, {items[1]}] -> {{{fmt_ar.split(':')[1]}}}"
    if pos: en += f", {pos[0]}"; ar += f" {pos[1]}"
    return {"english": en, "artha": ar}

def gen_translate():
    lang_en,lang_ar = r(LANGS)
    tone_en,tone_ar = r(TONES)
    words = r([100,150,200])
    return {"english": f"{filler()}translate this text to {lang_en}, keep it {tone_en}, under {words} words",
            "artha": f"tns[txt]({lang_ar}, {tone_ar}) <{words}w"}

def gen_email():
    aud_en,aud_ar = r(AUDIENCES)
    tone_en,tone_ar = r(TONES)
    words = r([100,150,200])
    return {"english": f"{filler()}write a {tone_en} email to {aud_en}, under {words} words",
            "artha": f"gen[eml]({aud_ar}, {tone_ar}, ~{words}w)"}

def gen_review():
    lang_en,lang_ar = r(CODE_LANGS)
    pos = maybe(CONSTRAINTS_POS, 0.8)
    neg = maybe(CONSTRAINTS_NEG, 0.6)
    en = f"{filler()}review this {lang_en} code"
    ar = f"rev[code]({lang_ar})"
    constraints = []
    if pos: en += f", {pos[0]}"; constraints.append(pos[1])
    if neg: en += f", {neg[0]}"; constraints.append(neg[1])
    if constraints: ar += f" {' '.join(constraints)} -> {{report}}"
    return {"english": en, "artha": ar}

def gen_sequence():
    lang_en,lang_ar = r(CODE_LANGS)
    return {"english": f"{filler()}fix this {lang_en} code then document it",
            "artha": f"fix[code]({lang_ar}) >> doc[code] -> {{md}}"}

GENERATORS = [(gen_summarise,0.18),(gen_generate,0.15),(gen_fix,0.15),
              (gen_explain,0.15),(gen_compare,0.12),(gen_translate,0.08),
              (gen_email,0.08),(gen_review,0.07),(gen_sequence,0.02)]

print('Generating 50,000 pairs...')
corpus = []
gens, weights = zip(*GENERATORS)
for _ in range(50000):
    fn = random.choices(gens, weights=weights, k=1)[0]
    try: corpus.append(fn())
    except: pass

with open('/content/data/corpus.jsonl', 'w') as f:
    for item in corpus:
        f.write(json.dumps(item) + '\n')

avg_en = sum(len(p['english'].split()) for p in corpus) / len(corpus)
avg_ar = sum(len(p['artha'].split()) for p in corpus) / len(corpus)
print(f'Generated {len(corpus)} pairs')
print(f'Avg English: {avg_en:.1f} words')
print(f'Avg Artha:   {avg_ar:.1f} words')
print(f'Compression: {round((1-avg_ar/avg_en)*100)}%')

Generating 50,000 pairs...
Generated 50000 pairs
Avg English: 12.0 words
Avg Artha:   3.8 words
Compression: 68%


In [4]:
# Cell 4 — Load data and tokenizer
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
OUTPUT_DIR = '/content/data/artha-model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

pairs = []
with open('/content/data/corpus.jsonl') as f:
    for line in f:
        item = json.loads(line)
        pairs.append({'text': f"<|artha|>\n{item['english']}\n<|compress|>\n{item['artha']}<|end|>"})

dataset = Dataset.from_list(pairs)
split = dataset.train_test_split(test_size=0.05, seed=42)
print(f'Train: {len(split["train"])} | Eval: {len(split["test"])}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_special_tokens({'additional_special_tokens': ['<|artha|>', '<|compress|>', '<|end|>']})
print('Tokenizer ready. Vocab size:', len(tokenizer))

Train: 47500 | Eval: 2500


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Tokenizer ready. Vocab size: 32003


In [5]:
# Cell 5 — Tokenize dataset
def tokenize(example):
    result = tokenizer(example['text'], truncation=True, max_length=128, padding='max_length')
    result['labels'] = result['input_ids'].copy()
    return result

train_ds = split['train'].map(tokenize, batched=True, remove_columns=['text'])
eval_ds  = split['test'].map(tokenize, batched=True, remove_columns=['text'])
print('Tokenization complete.')

Map:   0%|          | 0/47500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Tokenization complete.


In [6]:
# Cell 6 — Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
print('Model loaded on:', next(model.parameters()).device)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded on: cpu


In [7]:
# Cell 7 — Apply LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,313,472 || trainable%: 0.2044


In [8]:
# Cell 8 — Train!
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    warmup_steps=50,
    learning_rate=2e-4,
    bf16=True,
    fp16=False,
    logging_steps=50,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to='none',
    optim='adamw_torch',
    dataloader_drop_last=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
)

print('Starting training...')
trainer.train()

Starting training...


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss
200,0.101250,0.096254
400,0.089375,0.090974
600,0.088750,0.086626
800,0.088750,0.086413
1000,0.085625,0.084920
1200,0.085000,0.085296
1400,0.085625,0.085045
1600,0.082500,0.083962
1800,0.083750,0.083812
2000,0.082500,0.084369


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:386: UserWarning: Setting `save_embedding_layers` to `True` as the embedding lay

TrainOutput(global_step=8904, training_loss=0.11997153666891285, metrics={'train_runtime': 2043.6984, 'train_samples_per_second': 69.709, 'train_steps_per_second': 4.357, 'total_flos': 1.1343558948264346e+17, 'train_loss': 0.11997153666891285, 'epoch': 3.0})

In [10]:
# Cell 9 — Save model (TPU-safe)
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Move model to CPU before saving
print("Moving model to CPU for saving...")
model = model.cpu()

# Save
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Model saved to", OUTPUT_DIR)

Moving model to CPU for saving...
Model saved to /content/data/artha-model


In [11]:
import os
files = os.listdir(OUTPUT_DIR)
print("Saved files:", files)

Saved files: ['checkpoint-8600', 'README.md', 'adapter_model.safetensors', 'checkpoint-8904', 'tokenizer.json', 'chat_template.jinja', 'tokenizer_config.json', 'adapter_config.json']


In [13]:
# Cell 10 — Test inference (TPU-safe)
from peft import PeftModel

# Load base model and resize to match training
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto'
)

# Must resize BEFORE loading adapter — matches training setup
tokenizer_inf = AutoTokenizer.from_pretrained(OUTPUT_DIR)
base.resize_token_embeddings(len(tokenizer_inf), mean_resizing=False)

# Load trained LoRA adapter
trained = PeftModel.from_pretrained(base, OUTPUT_DIR)
trained.eval()
print("Model loaded! Vocab size:", len(tokenizer_inf))

def compress(prompt):
    input_text = f'<|artha|>\n{prompt}\n<|compress|>\n'
    inputs = tokenizer_inf(input_text, return_tensors='pt').to(next(trained.parameters()).device)
    with torch.no_grad():
        outputs = trained.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer_inf.eos_token_id,
        )
    decoded = tokenizer_inf.decode(outputs[0], skip_special_tokens=False)
    return decoded.split('<|compress|>')[-1].split('<|end|>')[0].strip()

test_prompts = [
    'Please summarise this article in 3 bullet points, focus on key facts',
    'Fix the bug in this Python code and explain what was wrong',
    'Write a formal email to a client about the project delay, under 150 words',
    'Compare React and Vue for a beginner, format as table',
    'Explain machine learning to a 10 year old in simple language',
]

print('ARTHA COMPRESSION TEST')
print('=' * 60)
total_en, total_ar = 0, 0
for prompt in test_prompts:
    compressed = compress(prompt)
    en_t = len(prompt.split())
    ar_t = len(compressed.split())
    total_en += en_t
    total_ar += ar_t
    pct = round((1 - ar_t/en_t) * 100)
    print(f'EN ({en_t:2d}t): {prompt}')
    print(f'AR ({ar_t:2d}t): {compressed}')
    print(f'Saved: {pct}%')
    print()

print(f'Overall compression: {round((1-total_ar/total_en)*100)}%')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Model loaded! Vocab size: 32003
ARTHA COMPRESSION TEST
EN (12t): Please summarise this article in 3 bullet points, focus on key facts
AR ( 3t): sum[article](#3, fmt:bullets) +facts
Saved: 75%

EN (12t): Fix the bug in this Python code and explain what was wrong
AR ( 3t): fix[code](code:py) -> {diff+explain}
Saved: 75%

EN (14t): Write a formal email to a client about the project delay, under 150 words
AR ( 3t): gen[eml](@client, tone:formal, ~150w)
Saved: 79%

EN (10t): Compare React and Vue for a beginner, format as table
AR ( 4t): cmp[React, Vue] -> {table}
Saved: 60%

EN (11t): Explain machine learning to a 10 year old in simple language
AR ( 3t): xpl[machine learning](lvl:eli5, fmt:draft)
Saved: 73%

Overall compression: 73%


In [14]:
# Cell 11 — Download trained model
from google.colab import files
import shutil

shutil.make_archive('/content/artha-model', 'zip', OUTPUT_DIR)
files.download('/content/artha-model.zip')
print('Downloaded!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded!
